# 📊 Modelagem e Conclusões — Dengue em Recife

Este notebook apresenta os resultados da modelagem preditiva de casos de dengue
em Recife (PE) no período de 2015 a 2024, utilizando dados da API InfoDengue.

## 1. Carregando dados processados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

df = pd.read_csv("../data/processed/dengue_processado.csv")
df['data'] = pd.to_datetime(df['data'])

print(f"Total de semanas: {len(df)}")
print(f"Período: {df['data'].min().date()} a {df['data'].max().date()}")
print(f"Features disponíveis: {list(df.columns)}")
df.describe().round(1)

## 2. Principais achados da análise exploratória

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

outputs = [
    ("../outputs/casos_ao_longo_do_tempo.png", "Casos ao longo do tempo"),
    ("../outputs/niveis_alerta.png",            "Níveis de alerta"),
    ("../outputs/correlacao.png",               "Correlação variáveis"),
]

for ax, (path, title) in zip(axes, outputs):
    if os.path.exists(path):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Comparativo dos modelos

Três modelos foram treinados com split temporal (80% treino / 20% teste):
- **Regressão Linear** — baseline interpretável
- **Random Forest** — ensemble robusto
- **XGBoost** — gradient boosting otimizado

In [ ]:
resultados = {
    'Modelo':  ['Regressão Linear', 'Random Forest', 'XGBoost'],
    'MAE':     [21.9, 17.5, 14.3],
    'RMSE':    [30.6, 41.0, 38.1],
    'R²':      [0.976, 0.958, 0.964],
}
df_res = pd.DataFrame(resultados)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cores = ['#4C72B0', '#DD8452', '#55A868']

for ax, metrica in zip(axes, ['MAE', 'RMSE', 'R²']):
    bars = ax.bar(df_res['Modelo'], df_res[metrica], color=cores)
    ax.set_title(f'Comparativo — {metrica}')
    ax.set_xticklabels(df_res['Modelo'], rotation=15, ha='right')
    for bar, val in zip(bars, df_res[metrica]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/comparativo_modelos.png', dpi=150)
plt.show()

print(df_res.to_string(index=False))

## 4. Conclusão

O projeto demonstrou que é possível prever casos de dengue em Recife com alta
precisão usando dados históricos e variáveis climáticas.

**Principais conclusões:**

- O **XGBoost** obteve o menor erro absoluto (MAE = 14.3 casos/semana)
- A **Regressão Linear** surpreendeu com R² = 0.976, evidenciando a forte
  linearidade dos lag features
- Os **lag features** (casos nas semanas anteriores) foram as variáveis mais
  preditivas, seguidos pela média móvel de 4 semanas
- Recife apresenta **sazonalidade clara**: picos entre janeiro e abril,
  coincidindo com o período chuvoso

**Impacto potencial:**  
Um sistema de alerta baseado neste modelo poderia notificar a Secretaria de
Saúde com até 4 semanas de antecedência, permitindo ações preventivas antes
dos surtos.